In [ ]:
# Imports and configuration
import os
import math
import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML

# Backend base URL (set via CASE_MGMT_BACKEND_URL)
backendUrl = os.getenv('CASE_MGMT_BACKEND_URL', 'http://localhost:3000')
# Shared secret for proxy requests (set JUPYTER_SHARED_SECRET in environment)
JUPYTER_SHARED_SECRET = os.getenv('JUPYTER_SHARED_SECRET')
if JUPYTER_SHARED_SECRET:
    headers = {'x-jupyter-secret': JUPYTER_SHARED_SECRET}
else:
    headers = {}

# Fallback account ID (used when no account specified or when fetch returns 404)
FALLBACK_ACCOUNT_ID = 'dbtrAcct_de52b2c4041c4a03a73a92cfb0b8cf35'

# Account to visualize can be provided via env var ACCOUNT_ID, otherwise fallback used
accountId = os.getenv('ACCOUNT_ID', FALLBACK_ACCOUNT_ID)

# print(f'Using backend: {backendUrl}')
# print(f'Using accountId: {accountId}')


In [ ]:
# Fetch network analysis data with fallback on 404
def fetch_account_network(account_id, backend_url):
    url = f"{backend_url}/api/v1/jupyter/proxy/network-analysis/account/{account_id}"
    params = {'tenantId': 'DEFAULT'}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=15)
    except Exception as e:
        print('Request error:', e)
        return None

    # If not found and we're not already using the fallback, try fallback
    if resp.status_code == 404 and account_id != FALLBACK_ACCOUNT_ID:
        print(f'Network data not found for {account_id}, trying fallback: {FALLBACK_ACCOUNT_ID}')
        url = f"{backend_url}/api/v1/jupyter/proxy/network-analysis/account/{FALLBACK_ACCOUNT_ID}"
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=15)
        except Exception as e:
            print('Request error (fallback):', e)
            return None

    try:
        resp.raise_for_status()
    except Exception as e:
        print('HTTP error:', e)
        return None

    return resp.json()

# Example: fetch data now
data = fetch_account_network(accountId, backendUrl)
if not data:
    display(HTML(f"<div style='color:#ef4444'>No network data available for account: {accountId}</div>"))
# else:
#     print('Fetched network data keys:', list(data.keys()))


In [ ]:
# Normalize expected network payloads and build node/edge lists
nodes = []
edges = []

if data and isinstance(data, dict):
    network_data = data.get('network')
    if network_data and isinstance(network_data, dict):
        nodes = network_data.get('nodes', [])
        edges = network_data.get('edges', [])
    else:
        nodes = data.get('nodes', [])
        edges = data.get('edges', [])

nodes = [n for n in nodes if isinstance(n, dict)]
edges = [e for e in edges if isinstance(e, dict)]

# Build positions - circular layout with center node
def build_positions(nodes):
    positions = {}
    n = max(len(nodes), 1)
    center_idx = 0
    
    for i, node in enumerate(nodes):
        if node.get('isCenter'):
            center_idx = i
            break
    
    radius = 280
    angle_step = 2 * math.pi / max(n - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1
    return positions

positions = build_positions(nodes)

# Create edge traces with different colors based on transaction type
edge_traces = []
center_id = None
for node in nodes:
    if node.get('isCenter'):
        center_id = node.get('id')
        break

for e in edges:
    s = e.get('source') or e.get('from')
    t = e.get('target') or e.get('to')
    if s in positions and t in positions:
        x0, y0 = positions[s]
        x1, y1 = positions[t]
        
        # Determine edge color and style
        edge_color = '#94a3b8'  # default gray
        edge_dash = 'dash'
        if s == center_id:
            edge_color = '#f97316'  # orange for outbound from center
            edge_dash = 'solid'
        
        edge_trace = go.Scatter(
            x=[x0, x1, None],
            y=[y0, y1, None],
            mode='lines',
            line=dict(width=2, color=edge_color, dash=edge_dash),
            hoverinfo='none',
            showlegend=False
        )
        edge_traces.append(edge_trace)

# Build node traces
node_x = []
node_y = []
node_text = []
marker_size = []
marker_color = []
marker_line_color = []
marker_line_width = []
node_ids = []

def get_node_style(node):
    is_center = node.get('isCenter', False)
    status = node.get('status', 'normal')
    
    if is_center:
        return {
            'fill': '#fef3c7',
            'border': '#f59e0b',
            'size': 55,
            'line_width': 4
        }
    elif status in ('alert', 'flagged'):
        return {
            'fill': '#fee2e2',
            'border': '#ef4444',
            'size': 42,
            'line_width': 3
        }
    elif status == 'investigation':
        return {
            'fill': '#fef3c7',
            'border': '#f59e0b',
            'size': 42,
            'line_width': 3
        }
    else:
        return {
            'fill': '#e0e7ff',
            'border': '#6366f1',
            'size': 42,
            'line_width': 2
        }

for n in nodes:
    nid = n.get('id')
    if nid not in positions:
        continue
    
    x, y = positions[nid]
    node_x.append(x)
    node_y.append(y)
    
    label = n.get('label') or nid
    node_text.append(f"<b>{nid}</b><br>{label}")
    
    style = get_node_style(n)
    marker_color.append(style['fill'])
    marker_line_color.append(style['border'])
    marker_size.append(style['size'])
    marker_line_width.append(style['line_width'])
    node_ids.append(nid)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=[nid.split('-')[-1] if '-' in nid else nid[:8] for nid in node_ids],
    hovertext=node_text,
    hoverinfo='text',
    customdata=node_ids,
    marker=dict(
        color=marker_color,
        size=marker_size,
        line=dict(width=marker_line_width, color=marker_line_color)
    ),
    textposition='middle center',
    textfont=dict(size=9, color='#1e293b', family='Arial, sans-serif'),
    showlegend=False
)

# Calculate summary stats
linked_count = len([n for n in nodes if not n.get('isCenter')])
alert_count = len([n for n in nodes if n.get('status') in ('alert', 'flagged')])
total_tx = sum([e.get('txCount', 0) for e in edges])
total_value = sum([e.get('totalAmount', 0) for e in edges])

fig = go.Figure(
    data=edge_traces + [node_trace],
    layout=go.Layout(
        title=None,
        showlegend=False,
        hovermode='closest',
        margin=dict(b=60, l=60, r=60, t=60),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=600,
        paper_bgcolor='#ffffff',
        plot_bgcolor='#ffffff',
    )
)

if nodes:
    import json
    graph_id = 'network-graph-vis'
    graph_html = pio.to_html(fig, full_html=False, config={'displayModeBar': False}, div_id=graph_id)
    
    # Legend positioned at bottom-left
    legend_html = """
    <div style="position: absolute; bottom: 20px; left: 20px; background: white; padding: 16px 20px; border-radius: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.08); font-family: Arial, sans-serif; font-size: 12px; border: 1px solid #e2e8f0;">
        <div style="font-weight: 600; margin-bottom: 14px; color: #1e293b; font-size: 13px;">Legend</div>
        <div style="display: flex; align-items: center; margin-bottom: 10px;">
            <div style="width: 18px; height: 18px; border-radius: 50%; background: #e0e7ff; border: 2px solid #6366f1; margin-right: 10px;"></div>
            <span style="color: #475569;">Normal Account</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 10px;">
            <div style="width: 18px; height: 18px; border-radius: 50%; background: #fee2e2; border: 3px solid #ef4444; margin-right: 10px;"></div>
            <span style="color: #475569;">Alert Triggered</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 10px;">
            <div style="width: 18px; height: 18px; border-radius: 50%; background: #fef3c7; border: 4px solid #f59e0b; margin-right: 10px;"></div>
            <span style="color: #475569;">Center Account</span>
        </div>
        <div style="border-top: 1px solid #e2e8f0; margin: 12px 0 8px 0;"></div>
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <div style="width: 24px; height: 2px; background: #f97316; margin-right: 10px;"></div>
            <span style="color: #475569;">Outbound Flow</span>
        </div>
        <div style="display: flex; align-items: center;">
            <div style="width: 24px; height: 2px; background: #94a3b8; border-top: 2px dashed #94a3b8; margin-right: 10px;"></div>
            <span style="color: #475569;">Inbound Flow</span>
        </div>
    </div>
    """
    
    # Zoom and control buttons - positioned at top-right
    action_buttons_html = """
    <div style="position: absolute; top: 20px; right: 20px; display: flex; flex-direction: column; gap: 8px; z-index: 1000;">
        <button id="zoom-in-btn" style="width: 40px; height: 40px; background: white; border: 1px solid #e2e8f0; border-radius: 8px; cursor: pointer; display: flex; align-items: center; justify-content: center; box-shadow: 0 2px 4px rgba(0,0,0,0.08); transition: all 0.2s;" onmouseover="this.style.background='#f8fafc'" onmouseout="this.style.background='white'" title="Zoom In">
            <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="#475569" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
                <circle cx="11" cy="11" r="8"></circle>
                <line x1="11" y1="8" x2="11" y2="14"></line>
                <line x1="8" y1="11" x2="14" y2="11"></line>
                <line x1="21" y1="21" x2="16.65" y2="16.65"></line>
            </svg>
        </button>
        <button id="zoom-out-btn" style="width: 40px; height: 40px; background: white; border: 1px solid #e2e8f0; border-radius: 8px; cursor: pointer; display: flex; align-items: center; justify-content: center; box-shadow: 0 2px 4px rgba(0,0,0,0.08); transition: all 0.2s;" onmouseover="this.style.background='#f8fafc'" onmouseout="this.style.background='white'" title="Zoom Out">
            <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="#475569" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
                <circle cx="11" cy="11" r="8"></circle>
                <line x1="8" y1="11" x2="14" y2="11"></line>
                <line x1="21" y1="21" x2="16.65" y2="16.65"></line>
            </svg>
        </button>
        <button id="reset-view-btn" style="width: 40px; height: 40px; background: white; border: 1px solid #e2e8f0; border-radius: 8px; cursor: pointer; display: flex; align-items: center; justify-content: center; box-shadow: 0 2px 4px rgba(0,0,0,0.08); transition: all 0.2s;" onmouseover="this.style.background='#f8fafc'" onmouseout="this.style.background='white'" title="Reset View">
            <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="#475569" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
                <path d="M3 12a9 9 0 0 1 9-9 9.75 9.75 0 0 1 6.74 2.74L21 8"></path>
                <path d="M21 3v5h-5"></path>
                <path d="M21 12a9 9 0 0 1-9 9 9.75 9.75 0 0 1-6.74-2.74L3 16"></path>
                <path d="M3 21v-5h5"></path>
            </svg>
        </button>
        <button id="focus-node-btn" style="width: 40px; height: 40px; background: white; border: 1px solid #e2e8f0; border-radius: 8px; cursor: pointer; display: flex; align-items: center; justify-content: center; box-shadow: 0 2px 4px rgba(0,0,0,0.08); transition: all 0.2s;" onmouseover="this.style.background='#f8fafc'" onmouseout="this.style.background='white'" title="Focus on Selected Node">
            <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="#475569" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
                <circle cx="12" cy="12" r="3"></circle>
                <circle cx="12" cy="12" r="10"></circle>
                <line x1="12" y1="2" x2="12" y2="4"></line>
                <line x1="12" y1="20" x2="12" y2="22"></line>
                <line x1="2" y1="12" x2="4" y2="12"></line>
                <line x1="20" y1="12" x2="22" y2="12"></line>
            </svg>
        </button>
    </div>
    """
    
    # Get center account info
    center_account = next((n for n in nodes if n.get('isCenter')), {})
    account_details = data.get('accountDetails', {}) if isinstance(data, dict) else {}
    
    sidebar_html = f"""
    <div id="sidebar-panel" style="width: 280px; background: white; padding: 0; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.08); font-family: Arial, sans-serif; border: 1px solid #e2e8f0; overflow: hidden;">
        <!-- OVERVIEW STATE -->
        <div id="view-overview">
            <div style="background: #f8fafc; padding: 20px; border-bottom: 1px solid #e2e8f0;">
                <div style="font-size: 16px; font-weight: 700; color: #1e293b;">Center Account</div>
            </div>
            
            <div style="padding: 20px;">
                <div style="margin-bottom: 20px;">
                    <div style="font-size: 10px; color: #64748b; text-transform: uppercase; font-weight: 600; margin-bottom: 6px; letter-spacing: 0.5px;">ACCOUNT ID</div>
                    <div id="overview-id" style="font-size: 13px; color: #1e293b; font-weight: 600;">{center_account.get('id', '-')}</div>
                </div>
                
                <div style="margin-bottom: 20px;">
                    <div style="font-size: 10px; color: #64748b; text-transform: uppercase; font-weight: 600; margin-bottom: 6px; letter-spacing: 0.5px;">ACCOUNT HOLDER</div>
                    <div id="overview-name" style="font-size: 13px; color: #1e293b; font-weight: 600;">{center_account.get('label', '-')}</div>
                </div>

                <div style="border-top: 1px solid #e2e8f0; padding-top: 20px;">
                    <div style="font-size: 10px; font-weight: 700; color: #64748b; margin-bottom: 16px; text-transform: uppercase; letter-spacing: 0.5px;">NETWORK SUMMARY</div>
                    <div style="display: grid; gap: 12px;">
                        <div style="display: flex; justify-content: space-between; align-items: center;">
                            <span style="color: #64748b; font-size: 12px;">Connected Accounts:</span>
                            <span id="stat-linked" style="font-weight: 600; color: #1e293b; font-size: 13px;">{linked_count}</span>
                        </div>
                        <div style="display: flex; justify-content: space-between; align-items: center;">
                            <span style="color: #64748b; font-size: 12px;">Outbound Connections:</span>
                            <span id="stat-outbound" style="font-weight: 600; color: #1e293b; font-size: 13px;">{len([e for e in edges if (e.get('source') or e.get('from')) == center_id])}</span>
                        </div>
                        <div style="display: flex; justify-content: space-between; align-items: center;">
                            <span style="color: #64748b; font-size: 12px;">Inbound Connections:</span>
                            <span id="stat-inbound" style="font-weight: 600; color: #1e293b; font-size: 13px;">{len([e for e in edges if (e.get('target') or e.get('to')) == center_id])}</span>
                        </div>
                        <div style="display: flex; justify-content: space-between; align-items: center;">
                            <span style="color: #64748b; font-size: 12px;">Accounts with Alerts:</span>
                            <span id="stat-alerts" style="font-weight: 600; color: #ef4444; font-size: 13px;">{alert_count}</span>
                        </div>
                    </div>
                </div>
            </div>
        </div>

        <!-- DETAILS STATE -->
        <div id="view-details" style="display: none;">
            <div style="background: #f8fafc; padding: 20px; border-bottom: 1px solid #e2e8f0;">
                <div id="detail-title" style="font-size: 16px; font-weight: 700; color: #1e293b;">Connected Account</div>
            </div>
            
            <div style="padding: 20px;">
                <div style="margin-bottom: 20px;">
                    <div style="font-size: 10px; color: #64748b; text-transform: uppercase; font-weight: 600; margin-bottom: 6px; letter-spacing: 0.5px;">ACCOUNT ID</div>
                    <div id="detail-id" style="font-size: 13px; color: #1e293b; font-weight: 600; word-break: break-all;">-</div>
                </div>
                
                <div style="margin-bottom: 20px;">
                    <div style="font-size: 10px; color: #64748b; text-transform: uppercase; font-weight: 600; margin-bottom: 6px; letter-spacing: 0.5px;">ACCOUNT HOLDER</div>
                    <div id="detail-holder" style="font-size: 13px; color: #1e293b; font-weight: 600;">-</div>
                </div>
                
                <div style="margin-bottom: 24px;">
                    <div style="font-size: 10px; color: #64748b; text-transform: uppercase; font-weight: 600; margin-bottom: 6px; letter-spacing: 0.5px;">STATUS</div>
                    <div id="detail-status" style="display: inline-block; padding: 4px 12px; border-radius: 12px; font-size: 12px; font-weight: 600;">-</div>
                </div>

                <div style="border-top: 1px solid #e2e8f0; padding-top: 20px; margin-bottom: 20px;">
                    <div style="font-size: 10px; font-weight: 700; color: #64748b; margin-bottom: 16px; text-transform: uppercase; letter-spacing: 0.5px;">CONNECTIONS</div>
                    <div style="display: grid; gap: 12px;">
                        <div style="display: flex; justify-content: space-between; align-items: center;">
                            <span style="color: #64748b; font-size: 12px;">Outbound:</span>
                            <span id="detail-outbound" style="font-weight: 600; color: #1e293b; font-size: 13px; text-align: right;">0</span>
                        </div>
                        <div style="display: flex; justify-content: space-between; align-items: center;">
                            <span style="color: #64748b; font-size: 12px;">Inbound:</span>
                            <span id="detail-inbound" style="font-weight: 600; color: #1e293b; font-size: 13px; text-align: right;">0</span>
                        </div>
                    </div>
                </div>
                
                <div style="text-align: center; padding-top: 12px; border-top: 1px solid #e2e8f0;">
                    <button onclick="resetSidebar()" style="background: white; border: 1px solid #cbd5e1; color: #64748b; padding: 8px 16px; border-radius: 6px; font-size: 12px; cursor: pointer; font-weight: 500; transition: all 0.2s; width: 100%;">← Back to Overview</button>
                </div>
                
                <div style="margin-top: 12px;">
                    <button id="focus-from-sidebar" style="width: 100%; padding: 10px; background: #3b82f6; color: white; border: none; border-radius: 6px; font-size: 12px; font-weight: 600; cursor: pointer; transition: all 0.2s;" onmouseover="this.style.background='#2563eb'" onmouseout="this.style.background='#3b82f6'">
                        Focus on This Account
                    </button>
                </div>
            </div>
        </div>
    </div>
    """
    
    interactive_script = f"""
    <script>
    (function() {{
        const nodes = {json.dumps(nodes)};
        const edges = {json.dumps(edges)};
        const positions = {json.dumps(positions)};
        const centerAcct = {json.dumps(center_account)};
        const graphDiv = document.getElementById('{graph_id}');
        
        let selectedNodeId = centerAcct ? centerAcct.id : null;
        let currentZoomLevel = 1.0;
        
        window.resetSidebar = function() {{
            document.getElementById('view-overview').style.display = 'block';
            document.getElementById('view-details').style.display = 'none';
            selectedNodeId = centerAcct ? centerAcct.id : null;
        }};
        
        function updateSidebar(nodeId) {{
            selectedNodeId = nodeId;
            const node = nodes.find(n => n.id === nodeId);
            if (!node) return;
            
            document.getElementById('view-overview').style.display = 'none';
            document.getElementById('view-details').style.display = 'block';
            
            const title = node.isCenter ? 'Center Account' : 'Connected Account';
            document.getElementById('detail-title').innerText = title;
            document.getElementById('detail-id').innerText = node.id;
            document.getElementById('detail-holder').innerText = node.label || node.id;
            
            const statusEl = document.getElementById('detail-status');
            const status = node.status || 'normal';
            
            if (status === 'alert' || status === 'flagged') {{
                statusEl.innerText = 'ALERT';
                statusEl.style.background = '#fee2e2';
                statusEl.style.color = '#dc2626';
            }} else if (status === 'investigation') {{
                statusEl.innerText = 'UNDER INVESTIGATION';
                statusEl.style.background = '#fef3c7';
                statusEl.style.color = '#d97706';
            }} else {{
                statusEl.innerText = 'NORMAL';
                statusEl.style.background = '#f0f9ff';
                statusEl.style.color = '#0369a1';
            }}
            
            let outbound = 0, inbound = 0;
            edges.forEach(e => {{
                const src = e.source || e.from;
                const tgt = e.target || e.to;
                if (src === nodeId) outbound++;
                if (tgt === nodeId) inbound++;
            }});
            
            document.getElementById('detail-outbound').innerText = outbound;
            document.getElementById('detail-inbound').innerText = inbound;
        }}
        
        function zoomIn() {{
            currentZoomLevel *= 1.3;
            applyZoom();
        }}
        
        function zoomOut() {{
            currentZoomLevel /= 1.3;
            applyZoom();
        }}
        
        function resetZoom() {{
            currentZoomLevel = 1.0;
            const update = {{
                'xaxis.range': [-350, 350],
                'yaxis.range': [-350, 350]
            }};
            Plotly.relayout(graphDiv, update);
        }}
        
        function applyZoom() {{
            const range = 350 / currentZoomLevel;
            const update = {{
                'xaxis.range': [-range, range],
                'yaxis.range': [-range, range]
            }};
            Plotly.relayout(graphDiv, update);
        }}
        
        function focusOnNode() {{
            if (!selectedNodeId || !positions[selectedNodeId]) return;
            
            const pos = positions[selectedNodeId];
            const zoomLevel = 2.5;
            const range = 150;
            
            const update = {{
                'xaxis.range': [pos[0] - range/zoomLevel, pos[0] + range/zoomLevel],
                'yaxis.range': [pos[1] - range/zoomLevel, pos[1] + range/zoomLevel]
            }};
            
            Plotly.relayout(graphDiv, update);
        }}
        
        if (graphDiv) {{
            graphDiv.on('plotly_click', d => {{
                if (d.points && d.points[0] && d.points[0].customdata) {{
                    updateSidebar(d.points[0].customdata);
                }}
            }});
        }}
        
        document.getElementById('zoom-in-btn').addEventListener('click', zoomIn);
        document.getElementById('zoom-out-btn').addEventListener('click', zoomOut);
        document.getElementById('reset-view-btn').addEventListener('click', resetZoom);
        document.getElementById('focus-node-btn').addEventListener('click', focusOnNode);
        
        const focusSidebarBtn = document.getElementById('focus-from-sidebar');
        if (focusSidebarBtn) {{
            focusSidebarBtn.addEventListener('click', focusOnNode);
        }}
    }})();
    </script>
    """
    
    container_html = f"""
    <div style="display: flex; gap: 24px; background: #f1f5f9; padding: 24px; border-radius: 16px;">
        <div style="flex: 1; background: white; border-radius: 12px; border: 1px solid #e2e8f0; position: relative; overflow: hidden;">
            {graph_html}
            {action_buttons_html}
            {legend_html}
        </div>
        {sidebar_html}
    </div>
    {interactive_script}
    """
    
    display(HTML(container_html))
else:
    display(HTML('<div style="color: #ef4444; padding: 20px;">No network data available</div>'))